# VisionGuard: experiment notebook
**Adversarial robustness of post-training-quantised Vision Transformer and ConvNeXt models (FP32 / FP16 / INT8)**

Run top to bottom in Google Colab (**Runtime → Change runtime type → GPU**, e.g. A100; tick *High-RAM* if offered). The experiment code lives in `visionguard.py`; this notebook only configures and runs it.

| Section | What happens |
|---|---|
| 1. Setup | Install packages, mount Google Drive, load the library |
| 2. Configuration | Choose `MODE` (`quick` dry run or `full` study) and the Drive results folder |
| 3. Run the experiment | Per dataset × architecture: fine-tune → Gate 1 → calibrate and quantise → Gate 2 → attacks → diagnostics → efficiency |
| 4. Latency | Optional: re-measure latency for all pairs in one session |
| 5. Analysis | Attack success rates with bootstrap CIs, McNemar tests, tipping points, diagnostics D1–D5, plots |
| 6. Additional checks | FP16 gradient-underflow check and a stronger-attack check (APGD-CE) |
| 7. Optional extras | Multi-seed sensitivity study and a zip of all results |

**Resumable.** Results, fine-tuned weights and calibrated quantisation scales are written to Drive after every unit of work. After a disconnect, re-run Sections 1–2 and then Section 3; it continues where it stopped. Before re-running, check that Section 2 reports the existing results folder, otherwise a fresh run starts and fine-tunes every model again.

**Start with `MODE = "quick"`.** It exercises every stage on real models and data with 1 epoch and a few images; its timings let you estimate the full run.

## 1. Setup

### 1.1 Install packages and check the runtime

In [ ]:
# torchattacks pins an old "requests" that it never uses, so install it WITHOUT its dependency list
# (otherwise pip downgrades requests and floods the output with conflict warnings).
!pip -q install timm statsmodels datasets
!pip -q install --no-deps torchattacks
import torchattacks, timm; print("torchattacks", torchattacks.__version__, "| timm", timm.__version__)
import torch, os
print("torch", torch.__version__, "| CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available(): print(torch.cuda.get_device_name(0))
else: print("WARNING: no GPU - fine-tuning and attacks will be extremely slow. Change the runtime type.")
print("CPU cores (used for INT8 inference / latency):", os.cpu_count())

### 1.2 Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

### 1.3 Load the library
Put `visionguard.py` (from the repository) next to this notebook, or upload it when asked.

In [ ]:
import os
if not os.path.exists("visionguard.py"):
    from google.colab import files
    print("Upload visionguard.py from the repository:")
    files.upload()

## 2. Configuration
`MODE = "full"` uses the paper's settings (15 epochs, 1,000 evaluation images per dataset, 512 calibration images, ε ∈ {1, 2, 3, 4, 6, 8, 12, 16}/255, PGD-20). `OUT_DIR` is where everything is saved and resumed from.

In [ ]:
import importlib, os
import visionguard as vg
importlib.reload(vg)

MODE = "full"                                             # "quick" = dry run (tens of minutes) | "full" = the actual study
OUT_DIR = "/content/drive/MyDrive/VisionGuard_results"    # results, weights and quantisation scales are saved here

cfg = vg.Config(out_dir=OUT_DIR)

if MODE == "quick":
    cfg.out_dir = OUT_DIR + "_quicktest"
    cfg.pairs = (("cifar100", "vit"), ("cifar100", "convnext"))
    cfg.epochs = 1
    cfg.eval_per_class = {"cifar100": 2, "tinyimagenet": 1}   # 200 images per dataset
    cfg.calib_n = 128
    cfg.eps_255 = (2, 8)
    cfg.pgd_steps = 10
    cfg.d5_n, cfg.d5_eps_255, cfg.k50_n, cfg.k50_eps_255 = 50, (8,), 20, (8,)
    cfg.n_boot, cfg.warmup, cfg.iters = 200, 5, 20
    cfg.strict_gates = False       # report gate failures instead of stopping the dry run
else:
    # Full run: the paper's settings
    cfg.epochs = 15
    cfg.eval_per_class = {"cifar100": 10, "tinyimagenet": 5}  # 1,000 images per dataset
    cfg.calib_n = 512
    cfg.eps_255 = (1, 2, 3, 4, 6, 8, 12, 16)
    cfg.pgd_steps = 20
    cfg.gate2_agree = 0.95         # QSim / INT8 agreement threshold (relaxed from 0.99, disclosed in the paper)
    cfg.gate2_acc_tol = 0.03       # INT8 accuracy must stay within 3 pp of FP32
    cfg.strict_gates = True        # stop and explain if Gate 1 or Gate 2 fails
    cfg.pairs = None               # None = all four dataset × architecture pairs

### 2.1 Check the results folder before running
The folder should list the pair folders from earlier runs (or be empty on a first run).

In [ ]:
os.makedirs(cfg.out_dir, exist_ok=True)
existing = sorted(os.listdir(cfg.out_dir))
print("Drive mounted :", os.path.ismount("/content/drive"))
print("Results folder:", cfg.out_dir)
print("Existing items:", existing if existing else "EMPTY: a fresh run will fine-tune every model from scratch")
print("Pairs to run  :", cfg.all_pairs())
print("Environment   :", vg.env_info())

## 3. Run the experiment
One cell runs the whole grid. For each (dataset, architecture) pair it does:

1. **Fine-tune** FP32 (AdamW, cosine schedule, 5 % warm-up, no label smoothing, final epoch kept) → **Gate 1** (CIFAR-100 FP32 accuracy above 80 %).
2. **Calibrate** on 512 training images (histogram observers) and build the **INT8 model** (real integer kernels) and the **QSim** proxy sharing identical scales → **Gate 2** (INT8 accuracy within 3 pp of FP32, QSim/INT8 agreement ≥ 95 %, profiler shows `quantized::*` kernels).
3. **Attacks**: FGSM and PGD-20 for every ε plus the 64/255 diagnostic budget. Scenarios: A0 (FP32 white-box), A1 (crafted on QSim and scored on the real INT8 model; FP16 attacked in FP16), A2 (crafted on FP32, transferred to FP16 and INT8). Every adversarial image is checked to lie inside the ℓ∞ ball and [0, 1].
4. **Diagnostics**: D4 (zero-gradient fractions), D5 (margin-loss and temperature-scaled PGD, worst-case ASR) and the PGD K = 50 check.
5. **Efficiency**: serialised size, compression ratio and latency (50 warm-up and 200 timed iterations, batch 1 and 32).

If a gate fails in `full` mode the run stops with an explanation, so no attack results come from a broken setup.

In [ ]:
import time
t0 = time.time()
cache = {}
for ds, arch in cfg.all_pairs():
    if ds not in cache:                           # keep one dataset in RAM at a time
        cache.clear(); cache[ds] = vg.load_dataset_arrays(cfg, ds)   # downloads on first use, cached afterwards
        print(f"{ds}: train {len(cache[ds][0])}, test {len(cache[ds][1])}")
    t1 = time.time()
    ctx = vg.run_pipeline(cfg, ds, arch, data=cache[ds])
    print(f"=== {ds}/{arch} finished in {(time.time()-t1)/60:.1f} min ===")
    vg.free(ctx)
print(f"total {(time.time()-t0)/60:.1f} min")

### Troubleshooting
* **`GateError`** — the message names the gate and the cause. If FP32 accuracy is near chance, fine-tuning went wrong: move `fp32_final.pt` and `ft_checkpoint.pt` out of that pair's Drive folder (rename them rather than deleting) and re-run. If INT8 accuracy collapsed, inspect the pair with the snippet below.
* **Debugging one pair or stage:**
```python
data = vg.load_dataset_arrays(cfg, "cifar100")
ctx = vg.setup_context(cfg, "cifar100", "vit", data=data)   # loads or fine-tunes, then quantises
vg.clean_predictions(ctx); vg.gate2(ctx)
vg.run_attacks(ctx, kinds=("PGD",), eps_list=[8])           # a single budget
```

## 4. Latency *(optional)*
Latency depends on the CPU and GPU Colab assigns to a session. This cell re-measures efficiency for all pairs in one session, using the saved weights and calibrated scales (no re-training). Model sizes and compression ratios are unaffected. It regenerates each pair's `efficiency.csv`.

In [ ]:
for ds, arch in cfg.all_pairs():
    f = os.path.join(cfg.out_dir, f"{ds}_{arch}", "efficiency.csv")
    if os.path.exists(f):
        os.remove(f)               # regenerated below
    ctx = vg.setup_context(cfg, ds, arch)
    vg.measure_efficiency(ctx)
    vg.free(ctx)

## 5. Analysis
Everything below reads the saved per-sample predictions; no attacks are re-run. Tables are also written to `<results folder>/analysis/` as CSV and PNG.

### 5.1 Build the tables and show the validation gates

In [ ]:
import pandas as pd
from IPython.display import display, Image
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 30)
res = vg.analyse(cfg)
print("tables written to", os.path.join(cfg.out_dir, "analysis"))
print("\nGate 1"); display(res.get("gate1"))
print("Gate 2"); display(res.get("gate2")[["dataset","arch","acc_drop_fp32_int8","qsim_int8_agreement","cond_acc","cond_agree","cond_int_kernels","passed"]])

### 5.2 Attack success rate (PGD) by scenario and budget

In [ ]:
# ASR (%) on the matched set, PGD, by scenario and budget
a = res["asr"]
tab = a[a.attack == "PGD"].assign(asr=lambda d: (100 * d.asr).round(1)).pivot_table(
    index=["dataset", "arch", "scenario", "target"], columns="eps255", values="asr")
display(tab)

### 5.3 Tipping points
The budget ε* at which attack success crosses 25 %, 50 % and 75 %, and how much quantisation shifts it.

In [ ]:
# Does quantisation move the tipping point?  (eps* in x/255; NaN = threshold not reached within the grid)
t = res["tipping"]
display(t[t.note.isna()].pivot_table(index=["dataset","arch","attack","scenario","target"], columns="tau", values="eps_star_255").round(2))
print("Shift  eps*(FP32 A0) - eps*(compressed)  (positive = quantisation makes attacks succeed at a smaller budget)")
display(t[t.note.notna()].pivot_table(index=["dataset","arch","attack","scenario","target"], columns="tau", values="eps_star_255").round(2))

### 5.4 Paired significance tests
McNemar tests against the FP32 white-box attack, Holm-corrected across budgets.

In [ ]:
# Paired McNemar tests vs FP32 white-box, Holm-corrected across budgets
m = res["mcnemar"]
display(m[m.p_holm < 0.05][["dataset","arch","attack","eps255","compare","only_fp32","only_compressed","delta_asr","p_holm"]])
print(f"{(m.p_holm < 0.05).sum()} of {len(m)} comparisons significant after Holm correction")

### 5.5 Gradient-masking diagnostics (D1–D5, K = 50)

In [ ]:
# Gradient-masking diagnostics.  D1-D3 violations should be 0 or explainable; D4/D5 flag masking.
print("D1-D3"); display(res["checks"][res["checks"].violations > 0] if (res["checks"].violations > 0).any() else "no violations")
print("D4: fraction of samples with (near-)zero input gradient"); display(res.get("d4"))
print("D5: ASR under alternative losses; 'worst_case' = per-sample max over variants"); display(res.get("d5"))
print("K=20 vs K=50 PGD"); display(res.get("k50"))

### 5.6 Efficiency

In [ ]:
# Efficiency
display(res["eff"][["dataset","arch","precision","device","batch","median_ms","q25_ms","q75_ms","size_mb","compression_vs_fp32"]].round(2))

### 5.7 Attack-success curves with 95 % bootstrap CIs

In [ ]:
# ASR curves with 95% bootstrap CIs
import glob
for f in sorted(glob.glob(os.path.join(cfg.out_dir, "analysis", "asr_*.png"))):
    print(os.path.basename(f)); display(Image(f))

## 6. Additional checks
Two checks run after the main analysis. Both save a CSV to `<results folder>/analysis/`.

**6.1 FP16 gradient underflow.** Counts the pixels whose FP16 input gradient is exactly zero under three losses and compares PGD success on FP16 with FP32. With the default batch-mean loss most FP16 pixel gradients underflow to zero and the attack silently fails; loss scaling (built into `vg.HalfNet`) fixes this. The check bypasses the built-in scaling, so its `mean` rows show the unscaled behaviour.

In [ ]:
fp16_res = vg.run_over_pairs(cfg, vg.fp16_check, "fp16_check.csv")

**6.2 Stronger attack (APGD-CE).** Compares our PGD-20 with APGD-CE (50 steps), the main component of AutoAttack, on FP32 and INT8 at ε = 2 and 4/255, to test whether the INT8-versus-FP32 gap depends on the attack.

In [ ]:
apgd_res = vg.run_over_pairs(cfg, vg.apgd_check, "apgd_check.csv")
piv = apgd_res.pivot_table(index=["dataset", "arch", "eps255"], columns="target", values=["pgd20_ce", "apgd50_ce"])
piv[("gap_pp", "pgd20")] = (piv[("pgd20_ce", "A1_int8")] - piv[("pgd20_ce", "A0_fp32")]) * 100
piv[("gap_pp", "apgd50")] = (piv[("apgd50_ce", "A1_int8")] - piv[("apgd50_ce", "A0_fp32")]) * 100
display(piv.round(3))   # gap_pp = INT8 minus FP32 attack success, in percentage points

## 7. Optional extras
### 7.1 Multi-seed sensitivity study
Three seeds, each redrawing the calibration images and the PGD random start, at ε = 4 and 8/255 (mean and range). It re-calibrates and re-attacks, so it costs roughly three reduced runs.

In [ ]:
for ds, arch in cfg.all_pairs():
    print(ds, arch); vg.run_seed_study(cfg, ds, arch, data=vg.load_dataset_arrays(cfg, ds))

### 7.2 Bundle the results
Everything is already on Drive; this makes one zip to download.

In [ ]:
import shutil
from google.colab import files
zip_path = shutil.make_archive("/content/visionguard_results", "zip", cfg.out_dir)
print(zip_path)
files.download(zip_path)

---
### Notes
* **Quantisation tooling.** INT8 uses PyTorch's FX graph-mode flow (`prepare_qat_fx` for the calibrated fake-quant model, `convert_fx` for the integer model) on the oneDNN backend (the default x86/fbgemm path collapsed ConvNeXt INT8 on Colab CPUs). QSim and INT8 therefore share scales and zero-points, and real integer kernels run. Parts of `torch.ao.quantization` are being phased out, so keep the versions logged in `env.json` for every run.
* **What stays in float.** Resize and normalisation, attention matmuls and softmax, residual additions, class-token concatenation, layer-scale multiplications and depthwise convolutions; LayerNorm modules are excluded from quantisation by type (ConvNeXt's block layer norms are functional calls and still run as integer `quantized::layer_norm`). `gate2.json` lists the integer kernels the profiler actually saw.
* **Attention is un-fused in all precisions** (`TIMM_FUSED_ATTN=0`), so the graph is identical across FP32, FP16 and INT8 and traceable; the model is mathematically unchanged.
* **FP16** is evaluated on the GPU; FP32 and INT8 latency are measured on the same CPU; FP16 latency on the GPU is reported separately.
* **FP32 means FP32:** TF32 is switched off for evaluation and attacks.
* **FP16 attacks use loss scaling** (`FP16_LOSS_SCALE` in `visionguard.py`). Attack files created by an older version of the library crafted FP16 white-box attacks without it; move the `attacks/` folder of such a pair aside and re-run Section 3 to regenerate them.
* Per-sample predictions for every attack are stored in `<dataset>_<arch>/attacks/*.npz`, so any statistic can be recomputed without re-running attacks.